# Milestone 1 – Data Preparation
**Goal:** turn the raw Kaggle file into a clean, U.S.-only dataset of campaigns that either succeeded or failed, then prepare model-ready features.

Every step below prints a row count so the result can be checked and reproduced. The same filtering logic lives in `scripts/filter_campaigns.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "scripts"))
from filter_campaigns import US_STATES

raw = pd.read_csv(ROOT / "data/DSI_kickstarterscrape_dataset.csv", encoding="latin-1")
print(raw.shape)
raw.head(3)

(45957, 17)


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,138119,STRUM,http://www.kickstarter.com/projects/185476022/...,Film & Video,Animation,"Los Angeles, CA",live,20000.0,56.0,0.002800,3,"Fri, 08 Jun 2012 00:00:31 -0000",10,"$1,$10,$25,$40,$50,$100,$250,$1,000,$1,337,$9,001",1,0,28.00


## 1. Inspect the raw data

In [2]:
print(raw["status"].value_counts(), "\n")
missing = raw.isna().sum()
print("Missing values:\n", missing[missing > 0])

status
successful    22969
failed        18996
live           3929
canceled         59
suspended         4
Name: count, dtype: int64 

Missing values:
 location         1322
pledged            12
reward levels      59
dtype: int64


**Notes:** `live` campaigns have no final result yet, and `canceled`/`suspended` ones were stopped early, so none of them can be labeled success or fail. `location` has 1,322 blanks, which we can't place in a U.S. state.

## 2. Keep only campaigns with a final outcome

In [3]:
df = raw[raw["status"].isin(["successful", "failed"])].copy()
print(f"{len(raw):,} -> {len(df):,} rows")

45,957 -> 41,965 rows


### 2b. Fix inconsistent category labels
Some labels contain HTML codes, so `"Film &amp; Video"` and `"Film & Video"` show up as two different categories. `html.unescape` merges them.

In [4]:
import html
print("Before:", df["category"].nunique(), "categories, e.g.", [c for c in df["category"].unique() if "Film" in c])
for col in ["category", "subcategory", "location"]:
    df[col] = df[col].map(lambda s: html.unescape(s) if isinstance(s, str) else s)
print("After: ", df["category"].nunique(), "categories, e.g.", [c for c in df["category"].unique() if "Film" in c])
print(df["category"].nunique(), "categories,", df["subcategory"].nunique(), "subcategories")

Before: 14 categories, e.g. ['Film & Video', 'Film &amp; Video']
After:  13 categories, e.g. ['Film & Video']
13 categories, 49 subcategories


## 3. Keep U.S. campaigns and split location into city and state
U.S. locations look like `"Chicago, IL"`. Non-U.S. ones end in a country name (e.g. `"London, United Kingdom"`), so we keep rows whose last part is a valid 2-letter state code.

In [5]:
parts = df["location"].str.rsplit(",", n=1, expand=True)
df["city"] = parts[0].str.strip()
df["state"] = parts[1].str.strip()

before = len(df)
df = df[df["state"].isin(US_STATES)].copy()
print(f"{before:,} -> {len(df):,} rows  ({df['state'].nunique()} states incl. DC)")

41,965 -> 38,504 rows  (51 states incl. DC)


## 4. Handle remaining missing values

In [6]:
print(df.isna().sum()[lambda s: s > 0])
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"{before:,} -> {len(df):,} rows")

reward levels    41
dtype: int64
38,504 -> 38,463 rows


Only a few dozen rows had a missing `pledged` or `reward levels` value, so dropping them loses less than 0.2% of the data.

## 5. Create the target and check class balance

In [7]:
df["success"] = (df["status"] == "successful").astype(int)
df["success"].value_counts(normalize=True).round(3)

success
1    0.548
0    0.452
Name: proportion, dtype: float64

## 6. Remove data leakage
A model should only use information known **at launch**. These columns are only known after the campaign ends, so they would make results look falsely good:

| Column | Why it leaks |
|---|---|
| `pledged`, `funded percentage` | Directly decide success |
| `backers`, `comments` | Accumulate during the campaign |
| `funded date` | End date of the campaign |
| `status` | The label itself |

`updates` is a gray area: creators post updates during the campaign. I keep it for EDA but will test models **with and without** it in Milestone 3.

In [8]:
LEAKY = ["pledged", "funded percentage", "backers", "comments", "funded date", "status"]
ID_COLS = ["project id", "name", "url", "location"]
model_df = df.drop(columns=LEAKY + ID_COLS)
model_df.columns.tolist()

['category',
 'subcategory',
 'goal',
 'levels',
 'reward levels',
 'updates',
 'duration',
 'city',
 'state',
 'success']

## 7. Engineer features from reward levels
`reward levels` is a text list like `"$1,$5,$25,$100"`. I turn it into the lowest and highest reward amounts. (`levels` already gives the number of tiers.)

In [9]:
import re

def parse_rewards(text):
    # Split on commas that start a new '$' amount, so '$1,000' stays one value
    amounts = [float(a.replace(",", "")) for a in re.findall(r"\$([\d,]+(?:\.\d+)?)(?=,\$|$)", str(text))]
    return pd.Series({"min_reward": min(amounts) if amounts else np.nan,
                      "max_reward": max(amounts) if amounts else np.nan})

model_df[["min_reward", "max_reward"]] = model_df["reward levels"].apply(parse_rewards)
model_df = model_df.drop(columns="reward levels")
print(model_df["min_reward"].isna().sum(), "rows failed to parse")
model_df[["levels", "min_reward", "max_reward"]].describe().round(1)

0 rows failed to parse


,levels,min_reward,max_reward
count,38463.0,38463.0,38463.0
mean,8.0,8.9,1939.5
std,4.2,97.0,2732.0
min,1.0,1.0,1.0
25%,5.0,1.0,250.0
50%,7.0,5.0,1000.0
75%,10.0,10.0,2012.0
max,80.0,10000.0,10000.0


## 8. Encode categorical variables
- `category` (few values) and `state` → one-hot encoding
- `subcategory` (many values) → **frequency encoding** (how common it is) to avoid hundreds of columns
- `city` (thousands of values) → dropped for now; `state` captures location

In [10]:
print("Unique values:", {c: model_df[c].nunique() for c in ["category", "subcategory", "state", "city"]})

model_df["subcategory_freq"] = model_df["subcategory"].map(model_df["subcategory"].value_counts(normalize=True))
encoded = pd.get_dummies(model_df.drop(columns=["subcategory", "city"]),
                         columns=["category", "state"], drop_first=True, dtype=int)
print(encoded.shape)

Unique values: {'category': 13, 'subcategory': 49, 'state': 51, 'city': 3140}
(38463, 70)


## 9. Save outputs

In [11]:
out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "kickstarter_us_clean.csv", index=False)        # readable, for EDA
encoded.to_csv(out / "kickstarter_model_ready.csv", index=False)  # numeric, for modeling
print("Clean:", df.shape, " Model-ready:", encoded.shape)

Clean: (38463, 20)  Model-ready: (38463, 70)


## Transformation log
| Step | Rows after |
|---|---|
| Raw data | 45,957 |
| Keep successful / failed | 41,965 |
| Keep U.S. campaigns | 38,504 |
| Drop missing values | 38,463 |

Also fixed: HTML codes in category labels (`&amp;` → `&`), which merged duplicate categories.

The final count is close to the challenge's reference of ~38,491. The small gap comes from my stricter state-code check and dropping rows with missing `pledged`/`reward levels`.